# Southern California Bight — Surface Wind Animation
**Period:** November 18 – December 8, 2020  
**Data:** ERA5 hourly 10 m winds (u10, v10) via CDS API  
**Output:** `figs/scb_wind_animation.gif` and `figs/scb_wind_animation.mp4`

## 1. Download ERA5 wind data

In [ ]:
import cdsapi
import os

# Southern California Bight bounding box
# CDS area order: N, W, S, E
N, W, S, E = 35.0, -122.0, 32.0, -117.0

os.makedirs('era5_scb_data', exist_ok=True)
outfile = 'era5_scb_data/ERA5_winds_SCB_20201118_20201208.nc'

if os.path.exists(outfile):
    print(f'File already exists: {outfile}')
else:
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                '10m_u_component_of_wind',
                '10m_v_component_of_wind',
            ],
            'year':  '2020',
            'month': ['11', '12'],
            'day': [
                '18', '19', '20', '21', '22', '23', '24', '25',
                '26', '27', '28', '29', '30',          # November
                '01', '02', '03', '04', '05', '06', '07', '08',  # December
            ],
            'time': [f'{h:02d}:00' for h in range(24)],
            'area': [N, W, S, E],
            'data_format': 'netcdf',
            'download_format': 'unarchived',
        },
        outfile
    )
    print(f'Saved -> {outfile}')

## 2. Load and inspect the dataset

In [ ]:
import xarray as xr
import numpy as np

ds_raw = xr.open_dataset('era5_scb_data/ERA5_winds_SCB_20201118_20201208.nc')

# Determine the time coordinate name
time_coord = 'valid_time' if 'valid_time' in ds_raw.coords else 'time'

# Slice to exactly Nov 18 – Dec 8 2020
ds = ds_raw.sel({time_coord: slice('2020-11-18', '2020-12-08')})
print(ds)

times = ds[time_coord].values
import pandas as pd
times = pd.DatetimeIndex(times)
print(f'\nTime steps: {len(times)}')
print(f'First: {times[0]}')
print(f'Last:  {times[-1]}')

# Squeeze out any size-1 dims
u10 = ds['u10'].squeeze()
v10 = ds['v10'].squeeze()
lons = ds['longitude'].values
lats = ds['latitude'].values
print(f'\nLon range: {lons.min():.2f} to {lons.max():.2f}')
print(f'Lat range: {lats.min():.2f} to {lats.max():.2f}')
print(f'Wind speed range: {float(np.sqrt(u10**2 + v10**2).min()):.1f} – {float(np.sqrt(u10**2 + v10**2).max()):.1f} m/s')

## 3. Build the animation

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd

# ── Config ──────────────────────────────────────────────────────────────────
STRIDE   = 1          # spatial thinning (1 = all grid points)
FPS      = 8          # frames per second
DPI      = 100
VMAX     = 14         # wind speed colorbar max (m/s)
PROJ     = ccrs.PlateCarree()
EXTENT   = [W - 0.5, E + 0.5, S - 0.5, N + 0.5]

# ── Pre-compute wind speed ───────────────────────────────────────────────────
time_coord = 'valid_time' if 'valid_time' in ds.coords else 'time'
times = pd.DatetimeIndex(ds[time_coord].values)

U_all    = u10.values
V_all    = v10.values
SPEED_all = np.sqrt(U_all**2 + V_all**2)

lon_s = lons[::STRIDE]
lat_s = lats[::STRIDE]
LON2, LAT2 = np.meshgrid(lon_s, lat_s)

cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=0, vmax=VMAX)

# ── Set up figure ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(1, 1, 1, projection=PROJ)
ax.set_extent(EXTENT, crs=PROJ)

ax.add_feature(cfeature.LAND,      facecolor='#d0c8b0', zorder=3)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=4)
ax.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='gray', zorder=4)
ax.add_feature(cfeature.BORDERS,   linewidth=0.6, edgecolor='gray', zorder=4)

gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray',
                  alpha=0.5, linestyle='--')
gl.top_labels   = False
gl.right_labels = False

pcm = ax.pcolormesh(LON2, LAT2, SPEED_all[0, ::STRIDE, ::STRIDE],
                    cmap=cmap, norm=norm,
                    transform=PROJ, zorder=1, shading='auto')

scale_val = 150
qv = ax.quiver(
    LON2, LAT2,
    U_all[0, ::STRIDE, ::STRIDE],
    V_all[0, ::STRIDE, ::STRIDE],
    color='black', alpha=0.8,
    scale=scale_val, scale_units='width',
    width=0.003, headwidth=4, headlength=4,
    transform=PROJ, zorder=5
)

ax.quiverkey(qv, X=0.88, Y=0.04, U=5,
             label='5 m/s', labelpos='E', fontproperties={'size': 9})

cb = fig.colorbar(pcm, ax=ax, fraction=0.03, pad=0.04)
cb.set_label('Wind speed (m/s)', fontsize=10)

title = ax.set_title('', fontsize=11, fontweight='bold')

fig.text(0.5, 0.01,
         'ERA5 10 m winds  |  Southern California Bight  |  Nov 18 – Dec 8 2020',
         ha='center', fontsize=9, color='dimgray')

# ── Animation update ─────────────────────────────────────────────────────────
def update(i):
    pcm.set_array(SPEED_all[i, ::STRIDE, ::STRIDE].ravel())
    qv.set_UVC(U_all[i, ::STRIDE, ::STRIDE],
               V_all[i, ::STRIDE, ::STRIDE])
    title.set_text(times[i].strftime('%Y-%m-%d  %H:%M UTC'))
    return pcm, qv, title

n_frames = len(times)
print(f'Animating {n_frames} hourly frames ({times[0].date()} to {times[-1].date()}) at {FPS} fps ...')

ani = animation.FuncAnimation(
    fig, update,
    frames=n_frames,
    interval=1000 // FPS,
    blit=False
)

os.makedirs('figs', exist_ok=True)

# Save GIF
gif_path = 'figs/scb_wind_animation.gif'
ani.save(gif_path, writer='pillow', fps=FPS, dpi=DPI)
print(f'Saved GIF -> {gif_path}')

# Save MP4
mp4_path = 'figs/scb_wind_animation.mp4'
ani.save(mp4_path, writer=animation.FFMpegWriter(fps=FPS, bitrate=1800), dpi=DPI)
print(f'Saved MP4 -> {mp4_path}')

plt.close(fig)
print('Done.')

## 4. Preview a single frame (inline)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd

PREVIEW_FRAME = 0   # change to any index to inspect a different time step
VMAX_prev     = 14
scale_val     = 150

times_prev = pd.DatetimeIndex(ds[time_coord].values)
U_prev     = u10.values
V_prev     = v10.values
SPEED_prev = np.sqrt(U_prev**2 + V_prev**2)
LON2_prev, LAT2_prev = np.meshgrid(lons, lats)

EXTENT = [W - 0.5, E + 0.5, S - 0.5, N + 0.5]

fig2, ax2 = plt.subplots(1, 1, figsize=(9, 7),
                          subplot_kw={'projection': ccrs.PlateCarree()})
ax2.set_extent(EXTENT, crs=ccrs.PlateCarree())
ax2.add_feature(cfeature.LAND,      facecolor='#d0c8b0', zorder=3)
ax2.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=4)
ax2.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='gray', zorder=4)
gl2 = ax2.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5, linestyle='--')
gl2.top_labels = False; gl2.right_labels = False

pcm2 = ax2.pcolormesh(LON2_prev, LAT2_prev, SPEED_prev[PREVIEW_FRAME],
                       cmap=plt.cm.YlOrRd,
                       norm=mcolors.Normalize(0, VMAX_prev),
                       transform=ccrs.PlateCarree(), shading='auto', zorder=1)
qv2 = ax2.quiver(LON2_prev, LAT2_prev,
                 U_prev[PREVIEW_FRAME], V_prev[PREVIEW_FRAME],
                 color='black', alpha=0.8, scale=scale_val, scale_units='width',
                 width=0.003, headwidth=4, headlength=4,
                 transform=ccrs.PlateCarree(), zorder=5)
ax2.quiverkey(qv2, X=0.88, Y=0.04, U=5, label='5 m/s',
              labelpos='E', fontproperties={'size': 9})
fig2.colorbar(pcm2, ax=ax2, fraction=0.03, pad=0.04, label='Wind speed (m/s)')
ax2.set_title(times_prev[PREVIEW_FRAME].strftime('%Y-%m-%d %H:%M UTC'),
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figs/scb_wind_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figs/scb_wind_preview.png')

---
## 5. Download ERA5 precipitation data

In [ ]:
import cdsapi
import os

os.makedirs('era5_scb_data', exist_ok=True)
precip_file = 'era5_scb_data/ERA5_precip_SCB_20201118_20201208.nc'

if os.path.exists(precip_file):
    print(f'File already exists: {precip_file}')
else:
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': ['total_precipitation'],
            'year':  '2020',
            'month': ['11', '12'],
            'day': [
                '18', '19', '20', '21', '22', '23', '24', '25',
                '26', '27', '28', '29', '30',
                '01', '02', '03', '04', '05', '06', '07', '08',
            ],
            'time': [f'{h:02d}:00' for h in range(24)],
            'area': [N, W, S, E],
            'data_format': 'netcdf',
            'download_format': 'unarchived',
        },
        precip_file
    )
    print(f'Saved -> {precip_file}')

## 6. Load and inspect precipitation

In [ ]:
import xarray as xr
import numpy as np

ds_p_raw = xr.open_dataset('era5_scb_data/ERA5_precip_SCB_20201118_20201208.nc')

time_coord_p = 'valid_time' if 'valid_time' in ds_p_raw.coords else 'time'

# Slice to exactly Nov 18 – Dec 8 2020
ds_p = ds_p_raw.sel({time_coord_p: slice('2020-11-18', '2020-12-08')})

times_p = ds_p[time_coord_p].values
import pandas as pd
times_p = pd.DatetimeIndex(times_p)
print(f'Time steps: {len(times_p)}')
print(f'First: {times_p[0]}')
print(f'Last:  {times_p[-1]}')

# tp is in metres/hour; convert to mm/hr
tp = ds_p['tp'].squeeze() * 1000
print(f'\nPrecip range: {float(tp.min()):.3f} – {float(tp.max()):.3f} mm/hr')

lons_p = ds_p['longitude'].values
lats_p = ds_p['latitude'].values

## 7. Precipitation animation

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd

FPS_P = 8
DPI_P = 100
VMAX_P = 5.0   # mm/hr; adjust if storms are stronger

PROJ  = ccrs.PlateCarree()
EXTENT = [W - 0.5, E + 0.5, S - 0.5, N + 0.5]

times_p = pd.DatetimeIndex(ds_p[time_coord_p].values)
TP = tp.values   # shape: (time, lat, lon)

LON2_P, LAT2_P = np.meshgrid(lons_p, lats_p)

cmap_p = plt.cm.Blues
norm_p = mcolors.Normalize(vmin=0, vmax=VMAX_P)

fig_p = plt.figure(figsize=(9, 7))
ax_p  = fig_p.add_subplot(1, 1, 1, projection=PROJ)
ax_p.set_extent(EXTENT, crs=PROJ)

ax_p.add_feature(cfeature.LAND,      facecolor='#d0c8b0', zorder=3)
ax_p.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=4)
ax_p.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='gray', zorder=4)
ax_p.add_feature(cfeature.BORDERS,   linewidth=0.6, edgecolor='gray', zorder=4)

gl_p = ax_p.gridlines(draw_labels=True, linewidth=0.4, color='gray',
                       alpha=0.5, linestyle='--')
gl_p.top_labels   = False
gl_p.right_labels = False

pcm_p = ax_p.pcolormesh(LON2_P, LAT2_P, TP[0],
                         cmap=cmap_p, norm=norm_p,
                         transform=PROJ, zorder=1, shading='auto')

cb_p = fig_p.colorbar(pcm_p, ax=ax_p, fraction=0.03, pad=0.04)
cb_p.set_label('Precipitation (mm/hr)', fontsize=10)

title_p = ax_p.set_title('', fontsize=11, fontweight='bold')

fig_p.text(0.5, 0.01,
           'ERA5 total precipitation  |  Southern California Bight  |  Nov 18 – Dec 8 2020',
           ha='center', fontsize=9, color='dimgray')

def update_p(i):
    pcm_p.set_array(TP[i].ravel())
    title_p.set_text(times_p[i].strftime('%Y-%m-%d  %H:%M UTC'))
    return pcm_p, title_p

n_frames_p = len(times_p)
print(f'Animating {n_frames_p} hourly frames ({times_p[0].date()} to {times_p[-1].date()}) at {FPS_P} fps ...')

ani_p = animation.FuncAnimation(
    fig_p, update_p,
    frames=n_frames_p,
    interval=1000 // FPS_P,
    blit=False
)

os.makedirs('figs', exist_ok=True)

gif_p_path = 'figs/scb_precip_animation.gif'
ani_p.save(gif_p_path, writer='pillow', fps=FPS_P, dpi=DPI_P)
print(f'Saved GIF -> {gif_p_path}')

mp4_p_path = 'figs/scb_precip_animation.mp4'
ani_p.save(mp4_p_path, writer=animation.FFMpegWriter(fps=FPS_P, bitrate=1800), dpi=DPI_P)
print(f'Saved MP4 -> {mp4_p_path}')

plt.close(fig_p)
print('Done.')